# Probabilistic analysis — manuscript display items

Generates all probabilistic display items of the manuscript from the outputs of
the probabilistic pipeline (see README, Stages 1–6):

**Main text**
- `annual_3panel.png` (Fig. 2 — EAUA, CC, EARP triptych)
- `recovery_drivers_annual_vs_median.png` (Fig. 3 — recovery-driver scatterplots)
- `bivariate_map_B_risk_vs_capacity.png` (Fig. 4 — risk × construction capacity)

**Supplementary Information**
- `na_coast_hazard_overview.png` (wind/surge hazard overview)
- `median_event_3panel.png`, `max_event_3panel.png`
- `recovery_drivers_annual_max.png`
- `skewness_wd.png` (skewness of per-event damage)
- NRI construct validation: `table_nri_evaluation.tex`, `table_divergent_counties.tex`,
  `fig_eaua_vs_nri_eal.png`, `fig_recovery_nri_agreement_2x2.png`,
  `fig_recovery_resilience_agreement.png`, `fig_capacity_vs_resilience.png`,
  and the `nri_*.csv` result tables

**Required inputs** (all county-level; produced by the pipeline scripts)
- `analysis_output/county_event_frequency_damage_metrics.csv` (`analyze_event_frequency_damage.py`)
- `analysis_output/earp_per_county.csv` (`compute_recovery_potential.py`)
- `analysis_output/median_vs_max_event_comparison.csv` (`compare_median_vs_max_events.py`)
- `analysis_output/county_distribution_metrics.csv` (`analyze_recovery_distributions.py`)
- `data/recovery/recovery_potential.csv` (`run_pyrecodes_light.py`)
- `data/impact/per_event/` (Stage 4 impact CSVs)
- `data/hazard/*_ncep_reanal.mat`, `data/NRI_Table_Counties.csv` (external inputs, see README)

Figures are written to `figures/`, LaTeX tables to `tables/`, CSV results to `analysis_output/`.

In [ ]:
import os, sys, warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import spearmanr, pearsonr
from IPython.display import display

warnings.filterwarnings("ignore")

# -- repo root: works whether the kernel CWD is notebooks/ or the repo root --
_cwd = Path('.').resolve()
REPO = _cwd if (_cwd / 'data').exists() else _cwd.parent
if not (REPO / 'data').exists():
    raise RuntimeError(f'Cannot locate repo root from {_cwd}')

DATA_DIR    = REPO / "data"
OUTPUT_DIR  = REPO / "analysis_output"   # intermediate metrics (pipeline outputs)
FIGURES_DIR = REPO / "figures"           # manuscript figures
TABLES_DIR  = REPO / "tables"            # manuscript LaTeX tables
for _d in (OUTPUT_DIR, FIGURES_DIR, TABLES_DIR):
    _d.mkdir(exist_ok=True)
print(f'REPO = {REPO}')

DEFAULT_FREQ = 0.00067334  # events per year (Poisson rate)
RECOVERY_WEIGHTS = {"DS1": 1.0, "DS2": 1.0, "DS3": 3.0, "DS4": 6.0}

# ---------------------------------------------------------------------------
# Print dimensions (IOPP: 8.5 cm single / 15 cm double column) + font settings
# ---------------------------------------------------------------------------
W_SINGLE = 3.35   # inches — 8.5 cm, single column
W_DOUBLE = 5.91   # inches — 15.0 cm, double column

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})

# ---------------------------------------------------------------------------
# Configuration – visual style matching create_bivariate_maps.py
# ---------------------------------------------------------------------------

COASTAL_STATE_FIPS = [
    "01", "09", "10", "12", "13", "22", "23", "24", "25", "28",
    "33", "34", "36", "37", "42", "44", "45", "48", "51",
]

MAP_XLIM = (-107, -65)
MAP_YLIM = (24, 48)

NO_DATA_COLOR = "#bab9b9"
COUNTY_EDGECOLOR = "#aaaaaa"
COUNTY_LINEWIDTH = 0.1
STATE_EDGECOLOR = "#444444"
STATE_LINEWIDTH = 0.3

# ---------------------------------------------------------------------------
# Bivariate color grids (Figure 4)
# ---------------------------------------------------------------------------
# Map A (EARP × EAUA) — Steven's pink-purple-blue scheme
#   Row 0 = EARP tercile 1 (low EARP = high recovery, safe)
#   Row 2 = EARP tercile 3 (high EARP = low recovery, concerning)
#   Col 0 = EAUA tercile 1 (low risk), Col 2 = EAUA tercile 3 (high risk)
GRID_A = [
    ["#e8e8e8", "#ace4e4", "#5ac8c8"],
    ["#dfb0d6", "#a5add3", "#5698b9"],
    ["#be64ac", "#8c62aa", "#3b4994"],
]
# Map B (inv-CC × EAUA) — green scheme
#   Row 0 = inv_cc=1 (high capacity, safe), Row 2 = inv_cc=3 (low capacity, concerning)
GRID_B = [
    ["#e8e8e8", "#ace4e4", "#5ac8c8"],
    ["#b8d6be", "#90b2b3", "#567994"],
    ["#73ae80", "#5a9178", "#2a5a5b"],
]




def _save_fig(fig, stem: str):
    """Save figure as PNG (300 dpi) to FIGURES_DIR and display it inline."""
    fig.savefig(FIGURES_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    print(f"  Saved  {stem}.png  ->  {FIGURES_DIR}")
    display(fig)


## Map and scatter helpers

In [ ]:
def _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label,
                      norm=None, use_log=True, invert_cbar=False):
    """
    Draw a single choropleth panel on *ax* with a horizontal colorbar below.

    Parameters
    ----------
    gdf        : GeoDataFrame containing column *col* and geometry
    state_gdf  : GeoDataFrame used for state-border overlay
    ax         : matplotlib Axes to draw on
    col        : column name in *gdf* to plot
    cmap       : colormap string or object
    cbar_label : label for the colorbar
    norm       : matplotlib Norm; auto-computed from data if None
    use_log    : if True, use LogNorm (ignored when norm is supplied explicitly)
    """
    tmp = gdf[[col, "geometry"]].copy()
    tmp.loc[~(tmp[col] > 0), col] = np.nan

    valid = tmp[col].dropna()

    if norm is None and use_log and len(valid) > 0:
        vmin = valid.min()
        vmax = valid.max()
        if np.isfinite(vmin) and np.isfinite(vmax) and vmin > 0:
            norm = LogNorm(vmin=vmin / 2, vmax=vmax)

    # Colorbar axis slightly wider than the map panel
    cax = ax.inset_axes([0.125, -0.12, 0.75, 0.05])

    tmp.plot(
        column=col, cmap=cmap, norm=norm,
        edgecolor=COUNTY_EDGECOLOR, linewidth=COUNTY_LINEWIDTH,
        legend=True, ax=ax, cax=cax,
        legend_kwds={"orientation": "horizontal"},
        missing_kwds={
            "color": NO_DATA_COLOR,
            "edgecolor": COUNTY_EDGECOLOR,
            "linewidth": COUNTY_LINEWIDTH,
        },
    )

    # State borders overlay
    state_gdf.plot(ax=ax, facecolor="none",
                   edgecolor=STATE_EDGECOLOR, linewidth=STATE_LINEWIDTH)

    ax.set_xlim(MAP_XLIM)
    ax.set_ylim(MAP_YLIM)
    ax.set_aspect("equal")
    ax.axis("off")

    # Colorbar styling
    if invert_cbar:
        cax.invert_xaxis()
        cax.set_xticks([])
        cax.set_xlabel("Low     Recovery potential     High", fontsize=7, labelpad=3)
    else:
        cax.set_xlabel(cbar_label, fontsize=7, labelpad=2)
        cax.tick_params(labelsize=6)
        cax.tick_params(which="minor", length=0)
    for spine in cax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.5)


def _scatter_panel(ax, x, y, xlabel, ylabel, panel_label=None,
                   alpha=0.35, s=12, color="#2166ac"):
    """
    Draw a log-log scatter plot with Spearman ρ annotation.

    Parameters
    ----------
    ax           : matplotlib Axes
    x, y         : pandas Series (may contain NaN; zero values excluded)
    xlabel/ylabel: axis labels
    panel_label  : bold letter label in top-left corner
    """
    valid = (~x.isna()) & (~y.isna()) & (x > 0) & (y > 0)
    xv, yv = x[valid].values, y[valid].values

    ax.scatter(xv, yv, alpha=alpha, s=s, color=color, linewidths=0)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel, fontsize=7)
    ax.set_ylabel(ylabel, fontsize=7)
    ax.tick_params(axis="x", which="major", labelsize=6)
    ax.tick_params(axis="x", which="minor", length=0, width=0)
    ax.tick_params(axis="y", which="both", left=False, labelleft=False)

    for spine in ax.spines.values():
        spine.set_edgecolor("0.4")
        spine.set_linewidth(0.8)

    if len(xv) >= 3:
        r, _ = spearmanr(xv, yv)
        n = len(xv)
        ax.text(
            0.04, 0.96,
            f"ρ = {r:+.2f}\nn = {n:,}",
            transform=ax.transAxes, fontsize=6,
            va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.25", fc="white",
                      ec="0.7", alpha=0.85),
        )

    if panel_label is not None:
        ax.text(0.02, 0.98, f"{panel_label})", transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="top", ha="left")


def _color_scatter_panel(ax, x, y, c, xlabel, ylabel, clabel,
                          cmap, panel_label=None,
                          alpha=0.6, s=20, show_yticks=True):
    """
    Color-encoded log-log scatter with vertical colorbar and inverted y-axis.

    The y-axis is inverted so short recovery time (high RP) appears at the top.
    Correlation is Spearman ρ on log-transformed values.

    Parameters
    ----------
    ax           : matplotlib Axes
    x, y, c      : pandas Series – NaN and non-positive values are excluded
    xlabel/ylabel: axis labels
    clabel       : colorbar label
    cmap         : colormap string for the color variable
    panel_label  : letter rendered as "(a)" etc. in top-left corner
    show_yticks  : False on right panels to reduce clutter
    """
    valid = (
        (~x.isna()) & (~y.isna()) & (~c.isna())
        & (x > 0) & (y > 0) & (c > 0)
    )
    xv, yv, cv = x[valid].values, y[valid].values, c[valid].values

    sc = ax.scatter(
        xv, yv, c=cv, cmap=cmap, alpha=alpha, s=s,
        norm=LogNorm(vmin=cv.min(), vmax=cv.max()),
        linewidths=0,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=7)
    if show_yticks:
        ax.set_ylabel(ylabel, fontsize=7)
    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_edgecolor("0.4")
        spine.set_linewidth(0.8)
    ax.tick_params(color="0.4", labelcolor="0.2")

    ax.tick_params(axis="x", which="major", labelsize=6)
    ax.tick_params(axis="x", which="minor", length=0, width=0)
    ax.tick_params(axis="y", which="both", left=False, labelleft=False)

    # Side colorbar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(clabel, fontsize=7)
    cbar.ax.tick_params(which="both", labelsize=6)
    cbar.ax.tick_params(which="minor", length=0)
    for spine in cbar.ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.5)

    # Spearman ρ annotation
    if len(xv) >= 3:
        r, _ = spearmanr(xv, yv)
        corr_x = 0.05 if show_yticks else 0.35
        ax.text(
            corr_x, 0.02,
            f"\u03c1 = {r:+.2f}\nn = {len(xv):,}",
            transform=ax.transAxes, fontsize=6, va="bottom",
        )

    if panel_label is not None:
        ax.text(0.02, 0.98, f"{panel_label})", transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="top", ha="left")

    return cbar


## Load metrics and build the main county GeoDataFrame

In [ ]:
def load_spatial_data():
    """Load coastal county shapefile and derive state-boundary overlay."""
    print("  Loading county shapefile …")
    counties = gpd.read_file(DATA_DIR / "US_counties.shp")
    coastal = counties[counties["STATEFP"].isin(COASTAL_STATE_FIPS)].copy()
    coastal["GEOID"] = (coastal["STATEFP"] + coastal["COUNTYFP"]).str.zfill(5)
    state_gdf = coastal.dissolve(by="STATEFP").reset_index()
    print(f"    {len(coastal)} coastal counties, {len(state_gdf)} states")
    return coastal, state_gdf


def load_annual_metrics():
    """
    Load EAUA, EARP, and CC (all as fips-keyed DataFrames).

    Returns
    -------
    eaua_df : DataFrame with columns [fips, eaua]
    earp_df : DataFrame with columns [fips, earp]
    cc_df   : DataFrame with columns [fips, cc]
    """
    print("  Loading annual metrics …")

    # Expected Annual Units Affected (EAUA)
    dmg = pd.read_csv(OUTPUT_DIR / "county_event_frequency_damage_metrics.csv")
    dmg["fips"] = dmg["fips"].astype(str).str.zfill(5)
    dmg["eaua"] = dmg["total_weighted_damage_units"] * DEFAULT_FREQ
    eaua_df = dmg[["fips", "eaua"]].copy()

    # Expected Annual Recovery Potential (EARP; months/yr)
    earp_raw = pd.read_csv(OUTPUT_DIR / "earp_per_county.csv")
    earp_raw["fips"] = earp_raw["fips"].astype(str).str.zfill(5)
    earp_df = earp_raw[["fips", "earp_months_per_year"]].rename(
        columns={"earp_months_per_year": "earp"}
    )

    # Construction Capacity (CC; permits/month)
    permits = pd.read_csv(DATA_DIR / "selected_states_counties_with_permits.csv")
    permits["fips"] = permits["FIPS"].astype(str).str.zfill(5)
    permits["cc"] = permits["Average_Building_Permits(12 months)"]
    cc_df = permits[["fips", "cc"]].copy()

    print(f"    EAUA: {len(eaua_df)} counties")
    print(f"    EARP: {len(earp_df)} counties")
    print(f"    CC:   {len(cc_df)} counties")
    return eaua_df, earp_df, cc_df


def load_event_level_metrics():
    """
    Load per-county median and max event metrics.

    Uses the pre-computed CSV if available, otherwise raises an error
    (the CSV is produced by compare_median_vs_max_events.py).

    Returns
    -------
    DataFrame with columns: fips, median_weighted_damage, median_recovery_months,
                            max_weighted_damage, max_recovery_months
    """
    print("  Loading per-event metrics …")
    csv_path = OUTPUT_DIR / "median_vs_max_event_comparison.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{csv_path} not found. "
            "Run scripts/compare_median_vs_max_events.py first."
        )
    df = pd.read_csv(csv_path)
    df["fips"] = df["fips"].astype(str).str.zfill(5)
    print(f"    Median/max metrics: {len(df)} counties")
    return df


def load_distribution_metrics():
    """Load county-level distribution metrics (skewness etc.)."""
    print("  Loading distribution metrics …")
    df = pd.read_csv(OUTPUT_DIR / "county_distribution_metrics.csv")
    df["fips"] = df["fips"].astype(str).str.zfill(5)
    print(f"    Distribution metrics: {len(df)} counties")
    return df


def load_hazard_data():
    """
    Load wind and surge hazard matrices and compute per-county statistics.

    The wind matrix is (n_events × n_counties) = (5018 × 3220).
    County ordering follows county_region.csv (county_index column).

    Returns None if scipy is not available (S1 is then skipped).
    """
    print("  Loading hazard matrices …")
    try:
        from scipy.io import loadmat
    except ImportError:
        print("    scipy not found – skipping hazard data (Figure S1 unavailable)")
        return None

    county_map = pd.read_csv(DATA_DIR / "county_region.csv")
    county_map["fips"] = county_map["fips"].astype(str).str.zfill(5)
    n_map = len(county_map)

    wind_mat = loadmat(
        DATA_DIR / "hazard" / "maxwindmat_ncep_reanal.mat"
    )["maxwindmat"]  # (5018, 3220) events × counties

    surge_raw = loadmat(
        DATA_DIR / "hazard" / "maxelev_coastcounty_ncep_reanal.mat"
    )
    surge_key = "scounty_mhhw" if "scounty_mhhw" in surge_raw else "scounty"
    surge_mat = surge_raw[surge_key]  # may be (3220, 5018) or (5018, 3220)
    if surge_mat.shape[0] != wind_mat.shape[0]:
        surge_mat = surge_mat.T  # ensure (events, counties)

    rain_mat = loadmat(
        DATA_DIR / "hazard" / "ptot_rain_county_ncep_reanal.mat"
    )["ptot_mat"]  # (3220, 5018) counties × events
    # Ensure counties are on axis-0
    if rain_mat.shape[0] != wind_mat.shape[1]:
        rain_mat = rain_mat.T
    max_rain = rain_mat.max(axis=1)[:n_map]  # max across events per county

    max_wind = wind_mat.max(axis=0)[:n_map]
    mean_wind = wind_mat.mean(axis=0)[:n_map]
    max_surge = surge_mat.max(axis=0)[:n_map]
    pct_ts_wind = (wind_mat > 17.5).mean(axis=0)[:n_map] * 100

    hazard_df = county_map[["fips"]].copy()
    hazard_df["max_wind_ms"] = max_wind
    hazard_df["mean_wind_ms"] = mean_wind
    hazard_df["max_rain_mm"] = max_rain
    hazard_df["max_surge_m"] = max_surge
    hazard_df["pct_events_ts_wind"] = pct_ts_wind

    print(f"    Hazard data: {len(hazard_df)} counties")
    print(f"    Wind range:  {max_wind.min():.1f}–{max_wind.max():.1f} m/s")
    print(f"    Rain range:  {max_rain.min():.0f}–{max_rain.max():.0f} mm")
    print(f"    Surge range: {max_surge.min():.2f}–{max_surge.max():.2f} m")
    return hazard_df


def build_main_gdf(coastal_counties, eaua_df, earp_df, cc_df,
                   event_df, dist_df):
    """
    Merge all tabular metrics into the coastal county GeoDataFrame.

    Returns a GeoDataFrame with columns (in addition to shapefile attributes):
    eaua, earp, cc, median_weighted_damage, median_recovery_months,
    max_weighted_damage, max_recovery_months, wd_skew, rt_skew
    """
    # Combine tabular metrics
    metrics = (
        eaua_df
        .merge(earp_df, on="fips", how="outer")
        .merge(cc_df,   on="fips", how="outer")
    )

    event_cols = [
        "fips", "median_weighted_damage", "median_recovery_months",
        "max_weighted_damage", "max_recovery_months",
    ]
    metrics = metrics.merge(
        event_df[[c for c in event_cols if c in event_df.columns]],
        on="fips", how="outer",
    )

    dist_cols = ["fips", "wd_skew", "rt_skew"]
    metrics = metrics.merge(
        dist_df[[c for c in dist_cols if c in dist_df.columns]],
        on="fips", how="outer",
    )

    # Zero / negative → NaN for strictly positive metrics
    for col in ("eaua", "earp", "cc",
                "median_weighted_damage", "median_recovery_months",
                "max_weighted_damage", "max_recovery_months"):
        if col in metrics.columns:
            metrics.loc[~(metrics[col] > 0), col] = np.nan

    gdf = coastal_counties.merge(
        metrics, left_on="GEOID", right_on="fips", how="left"
    )
    return gdf


def _print_metric_summary(gdf, col, label):
    valid = gdf[col].dropna()
    valid = valid[valid > 0] if col not in ("wd_skew", "rt_skew") else valid.dropna()
    if len(valid) > 0:
        print(f"    {label:45s}  n={len(valid):4d}  "
              f"min={valid.min():.4g}  median={valid.median():.4g}  "
              f"max={valid.max():.4g}")


In [ ]:
coastal_counties, state_gdf = load_spatial_data()
eaua_df, earp_df, cc_df = load_annual_metrics()
event_df = load_event_level_metrics()
dist_df = load_distribution_metrics()
gdf = build_main_gdf(coastal_counties, eaua_df, earp_df, cc_df, event_df, dist_df)
print(f'Main GeoDataFrame: {len(gdf)} counties')


## Figure S1 — Hazard overview (`na_coast_hazard_overview.png`)

In [ ]:
def figS1_hazard_overview(coastal_counties, state_gdf, hazard_df):
    """
    Figure S1: Three-panel choropleth – maximum wind speed, maximum rainfall,
    and maximum surge height per county across all synthetic events.
    """
    print("\nFigure S1: Hazard overview …")

    if hazard_df is None:
        print("  Skipped (hazard data not available – run with climada_env)")
        return

    gdf = coastal_counties.merge(
        hazard_df, left_on="GEOID", right_on="fips", how="left"
    )

    fig, axes = plt.subplots(1, 3, figsize=(W_DOUBLE, 3.3))

    panels = [
        ("max_wind_ms",  "YlOrRd",  "Max wind speed [m s⁻¹]",       False),
        ("max_rain_mm",  "Blues",   "Max rainfall [mm]",           False),
        ("max_surge_m",  "viridis", "Max storm surge [m]",   False),
    ]
    labels = ["a", "b", "c"]

    for ax, (col, cmap, cbar_label, inv), lbl in zip(axes, panels, labels):
        _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label,
                          invert_cbar=inv, use_log=False)
        ax.set_title(f"{lbl})", loc="left", fontsize=8, fontweight="bold", pad=2)

    print("  Summary:")
    for col, _, label, *__ in panels:
        _print_metric_summary(gdf, col, label)

    plt.tight_layout(pad=0.5)
    _save_fig(fig, "na_coast_hazard_overview")
    plt.close()


hazard_df = load_hazard_data()
figS1_hazard_overview(coastal_counties, state_gdf, hazard_df)


## Figure 2 — Annual triptych (`annual_3panel.png`)

In [ ]:
def fig2_annual_triptych(gdf, state_gdf):
    """
    Figure 2: Three-panel choropleth showing EAUA, Construction Capacity,
    and EARP across the US Atlantic coast.
    """
    print("\nFigure 2: Annual triptych …")

    fig, axes = plt.subplots(1, 3, figsize=(W_DOUBLE, 3.3))

    panels = [
        ("eaua", "cividis",  "EAUA [weighted units yr⁻¹]",              False),
        ("cc",   "Greens",   "CC [permits month⁻¹]",  False),
        ("earp", "Purples_r",  "Recovery potential",                       True),
    ]
    labels = ["a", "b", "c"]

    for ax, (col, cmap, cbar_label, inv), lbl in zip(axes, panels, labels):
        _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label, invert_cbar=inv)
        ax.set_title(f"{lbl})", loc="left", fontsize=8, fontweight="bold", pad=2)

    print("  Summary:")
    for col, _, label, *__ in panels:
        _print_metric_summary(gdf, col, label)

    plt.tight_layout(pad=0.5)
    _save_fig(fig, "annual_3panel")
    plt.close()


fig2_annual_triptych(gdf, state_gdf)


## Figure 3 — Recovery drivers, annual vs median event (`recovery_drivers_annual_vs_median.png`)

In [ ]:
def fig3_recovery_drivers_scatter(gdf):
    """
    Figure 3: 2×2 color-encoded log-log scatter.
    Row 0 (Annual):       EARP vs EAUA / CC,   colored by CC / EAUA
    Row 1 (Median event): MRP  vs WUA  / CC,   colored by CC / WUA
    Y-axis inverted: short recovery time (high potential) at top.
    """
    print("\nFigure 3: Recovery drivers scatter (annual + median event) …")

    fig, axes = plt.subplots(2, 2, figsize=(W_DOUBLE, 4.2))

    # Row 0 – annual
    _color_scatter_panel(
        axes[0, 0], gdf["eaua"], gdf["earp"], gdf["cc"],
        xlabel="EAUA",
        ylabel="RP (low\u2013high)",
        clabel="CC",
        cmap="viridis",
        panel_label="a",
        show_yticks=True,
    )
    _color_scatter_panel(
        axes[0, 1], gdf["cc"], gdf["earp"], gdf["eaua"],
        xlabel="CC",
        ylabel="RP (low\u2013high)",
        clabel="EAUA",
        cmap="plasma",
        panel_label="b",
        show_yticks=False,
    )

    # Row 1 – median event
    _color_scatter_panel(
        axes[1, 0], gdf["median_weighted_damage"], gdf["median_recovery_months"],
        gdf["cc"],
        xlabel="WUA",
        ylabel="RP (low\u2013high)",
        clabel="CC",
        cmap="viridis",
        panel_label="c",
        show_yticks=True,
    )
    _color_scatter_panel(
        axes[1, 1], gdf["cc"], gdf["median_recovery_months"],
        gdf["median_weighted_damage"],
        xlabel="CC",
        ylabel="RP (low\u2013high)",
        clabel="WUA",
        cmap="plasma",
        panel_label="d",
        show_yticks=False,
    )

    # Row labels
    fig.text(0.02, 0.78, "annual", rotation=90, va="center", ha="center",
             fontsize=7, fontweight="bold")
    fig.text(0.02, 0.30, "median event", rotation=90, va="center", ha="center",
             fontsize=7, fontweight="bold")

    # Legend (no box)
    fig.text(
        0.9, 0.50,
        "EAUA = expected annual affected units\nWUA = weighted affected units\nCC = construction capacity\nRP = recovery potential",
        va="center", ha="left", fontsize=6,
    )

    plt.tight_layout(rect=[0.05, 0, 0.90, 1])
    _save_fig(fig, "recovery_drivers_annual_vs_median")
    plt.close()


fig3_recovery_drivers_scatter(gdf)


## Figures S5 and S6 — Median and maximum event triptychs (`median_event_3panel.png`, `max_event_3panel.png`)

In [ ]:
def figS5_median_event_triptych(gdf, state_gdf):
    """
    Figure S5: Three-panel choropleth – median per-event WUA, CC, MRP.
    """
    print("\nFigure S5: Median event triptych …")

    fig, axes = plt.subplots(1, 3, figsize=(W_DOUBLE, 3.3))

    panels = [
        ("median_weighted_damage", "cividis", "WUA [units]",              False),
        ("cc",                     "Greens",  "CC [permits month⁻¹]",  False),
        ("median_recovery_months", "Purples_r", "Recovery potential",                       True),
    ]
    labels = ["a", "b", "c"]

    for ax, (col, cmap, cbar_label, inv), lbl in zip(axes, panels, labels):
        _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label, invert_cbar=inv)
        ax.set_title(f"{lbl})", loc="left", fontsize=8, fontweight="bold", pad=2)

    print("  Summary:")
    for col, _, label, *__ in panels:
        _print_metric_summary(gdf, col, label)

    plt.tight_layout(pad=0.5)
    _save_fig(fig, "median_event_3panel")
    plt.close()


def figS6_max_event_triptych(gdf, state_gdf):
    """
    Figure S6: Three-panel choropleth – max per-event WUA, CC, max RP.
    """
    print("\nFigure S6: Max event triptych …")

    fig, axes = plt.subplots(1, 3, figsize=(W_DOUBLE, 3.3))

    panels = [
        ("max_weighted_damage",  "cividis", "WUA [units]",                False),
        ("cc",                   "Greens",  "CC [permits month⁻¹]",  False),
        ("max_recovery_months",  "Purples_r", "Recovery potential",                       True),
    ]
    labels = ["a", "b", "c"]

    for ax, (col, cmap, cbar_label, inv), lbl in zip(axes, panels, labels):
        _choropleth_panel(gdf, state_gdf, ax, col, cmap, cbar_label, invert_cbar=inv)
        ax.set_title(f"{lbl})", loc="left", fontsize=8, fontweight="bold", pad=2)

    print("  Summary:")
    for col, _, label, *__ in panels:
        _print_metric_summary(gdf, col, label)

    plt.tight_layout(pad=0.5)
    _save_fig(fig, "max_event_3panel")
    plt.close()


figS5_median_event_triptych(gdf, state_gdf)
figS6_max_event_triptych(gdf, state_gdf)


## Figure S7 — Recovery drivers, annual vs maximum event (`recovery_drivers_annual_max.png`)

In [ ]:
def figS7_annual_max_scatter(gdf):
    """
    Figure S7: Max-event recovery drivers – 1×2 color-encoded log-log scatter.
    (a) Max RP vs max WUA, colored by CC  (Greens)
    (b) Max RP vs CC,      colored by max WUA (cividis)
    Y-axis inverted: short recovery time (high potential) at top.
    """
    print("\nFigure S7: Max-event recovery drivers scatter …")

    fig, axes = plt.subplots(1, 2, figsize=(W_DOUBLE, 2.2))

    _color_scatter_panel(
        axes[0], gdf["max_weighted_damage"], gdf["max_recovery_months"], gdf["cc"],
        xlabel="WUA",
        ylabel="RP (low\u2013high)",
        clabel="CC",
        cmap="viridis",
        panel_label="a",
        show_yticks=True,
    )
    _color_scatter_panel(
        axes[1], gdf["cc"], gdf["max_recovery_months"], gdf["max_weighted_damage"],
        xlabel="CC",
        ylabel="RP (low\u2013high)",
        clabel="WUA",
        cmap="plasma",
        panel_label="b",
        show_yticks=False,
    )

    fig.text(0.02, 0.55, "maximum value", rotation=90, va="center", ha="center",
             fontsize=7, fontweight="bold")
    fig.text(
        0.95, 0.50,
        "WUA = weighted affected units\nCC = construction capacity\nRP = recovery potential",
        va="center", ha="left", fontsize=6,
    )

    plt.tight_layout(rect=[0.05, 0, 0.93, 1])
    _save_fig(fig, "recovery_drivers_annual_max")
    plt.close()


figS7_annual_max_scatter(gdf)


## Figure S8 — Skewness of per-event damage (`skewness_wd.png`)

In [ ]:
def _skewness_single_panel(gdf, state_gdf, col, cbar_label, stem):
    """
    Draw and save a single skewness choropleth with a right-side colorbar.
    """
    tmp = gdf[[col, "geometry"]].copy()
    valid = tmp[col].dropna()

    if len(valid) == 0:
        print(f"  No valid data for {col!r} – skipping")
        return

    vmax = float(np.percentile(valid.clip(lower=0), 95))
    norm = Normalize(vmin=0, vmax=max(vmax, 1e-6))

    fig, ax = plt.subplots(1, 1, figsize=(W_DOUBLE * 0.65, 3.3))

    # No-data counties
    tmp[tmp[col].isna()].plot(
        ax=ax, color=NO_DATA_COLOR,
        edgecolor=COUNTY_EDGECOLOR, linewidth=COUNTY_LINEWIDTH,
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.1)

    tmp[tmp[col].notna()].plot(
        column=col, cmap="YlOrRd", norm=norm,
        edgecolor=COUNTY_EDGECOLOR, linewidth=COUNTY_LINEWIDTH,
        legend=True, ax=ax, cax=cax,
        legend_kwds={"orientation": "vertical"},
    )

    state_gdf.plot(ax=ax, facecolor="none",
                   edgecolor=STATE_EDGECOLOR, linewidth=STATE_LINEWIDTH)

    ax.set_xlim(MAP_XLIM)
    ax.set_ylim(MAP_YLIM)
    ax.set_aspect("equal")
    ax.axis("off")

    cax.set_ylabel(cbar_label, fontsize=7, labelpad=3)
    cax.tick_params(labelsize=6)
    cax.tick_params(which="minor", length=0)
    for spine in cax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.5)

    plt.tight_layout(pad=0.3)
    _save_fig(fig, stem)
    plt.close()

    v = gdf[col].dropna()
    print(f"    {cbar_label:55s}  n={len(v):4d}  "
          f"median={v.median():.2f}  p95={np.percentile(v, 95):.2f}  "
          f"max={v.max():.2f}")


print("  Summary:")
_skewness_single_panel(
    gdf, state_gdf,
    col="wd_skew",
    cbar_label="Skewness",
    stem="skewness_wd",
)


## Figure 4 — Bivariate map, risk × construction capacity (`bivariate_map_B_risk_vs_capacity.png`)

In [ ]:
def _bv_assign_tercile(series):
    """Assign tercile labels 1/2/3 (float); non-positive / non-finite → NaN."""
    valid = series.notna() & np.isfinite(series) & (series > 0)
    result = pd.Series(np.nan, index=series.index, dtype=float)
    if valid.sum() < 3:
        return result
    try:
        labels = pd.qcut(series[valid], q=3, labels=[1, 2, 3], duplicates="drop")
    except ValueError:
        labels = pd.cut(series[valid], bins=3, labels=[1, 2, 3])
    result[valid] = labels.astype(float)
    return result


def _bv_get_color(row, row_col, col_col, grid):
    t_row, t_col = row[row_col], row[col_col]
    if pd.isna(t_row) or pd.isna(t_col):
        return NO_DATA_COLOR
    return grid[int(t_row) - 1][int(t_col) - 1]


def _bv_prepare_gdf(coastal_counties, eaua_df, earp_df, cc_df):
    """Merge metrics into coastal county GDF and compute bivariate colors."""
    metrics = (
        eaua_df
        .merge(earp_df, on="fips", how="outer")
        .merge(cc_df,   on="fips", how="outer")
    )
    for col in ("eaua", "earp", "cc"):
        metrics.loc[~(metrics[col] > 0), col] = np.nan

    metrics["eaua_tercile"] = _bv_assign_tercile(metrics["eaua"])
    metrics["earp_tercile"] = _bv_assign_tercile(metrics["earp"])
    metrics["cc_tercile"]   = _bv_assign_tercile(metrics["cc"])

    gdf = coastal_counties.merge(metrics, left_on="GEOID", right_on="fips", how="left")

    gdf["color_a"] = gdf.apply(
        lambda r: _bv_get_color(r, "earp_tercile", "eaua_tercile", GRID_A), axis=1
    )
    # Invert CC so high CC (safe) → row 0, low CC (risky) → row 2
    gdf["inv_cc_tercile"] = 4 - gdf["cc_tercile"]
    gdf["color_b"] = gdf.apply(
        lambda r: _bv_get_color(r, "inv_cc_tercile", "eaua_tercile", GRID_B), axis=1
    )
    return gdf


def _bv_legend(ax, colors_display, xlabel, ylabel,
               x_low="Low", x_high="High", y_low="Low", y_high="High",
               bbox=(0.72, 0.01, 0.27, 0.34),
               xlabel_pos=(2.5, -1.8),
               ylabel_pos=(-2.2, 1.3),
               tick_spread=0.4,
               tick_offset=1.1):
    """
    Draw a 3×3 bivariate colour legend as an inset on *ax*.

    Label tuning parameters
    -----------------------
    xlabel_pos   : (x, y) data-coords for the x-axis name (e.g. "Risk")
    ylabel_pos   : (x, y) data-coords for the y-axis name (e.g. "CC")
    tick_spread  : how far Low/High labels extend beyond the arrow endpoints
                   (arrow runs 0→3; spread=0.4 → Low at -0.4, High at 3.4)
    tick_offset  : perpendicular distance of Low/High labels from the arrow line
    """
    axins = ax.inset_axes(bbox)
    for row_idx in range(3):
        for col_idx in range(3):
            rect = mpatches.Rectangle(
                (col_idx, row_idx), 1, 1,
                facecolor=colors_display[row_idx][col_idx],
                edgecolor="white", linewidth=0.5,
                transform=axins.transData,
            )
            axins.add_patch(rect)
    axins.set_xlim(-1.0, 3.5)
    axins.set_ylim(-1.0, 3.5)
    axins.set_aspect("equal")
    axins.axis("off")

    kw = dict(arrowstyle="->", color="black", lw=0.8)
    axins.annotate("", xy=(3.0, -0.5), xytext=(0.0, -0.5),
                   xycoords="data", textcoords="data",
                   arrowprops=kw, annotation_clip=False)
    axins.annotate("", xy=(-0.5, 3.0), xytext=(-0.5, 0.0),
                   xycoords="data", textcoords="data",
                   arrowprops=kw, annotation_clip=False)

    axins.text(*xlabel_pos, xlabel, ha="center", va="top",
               fontsize=7, transform=axins.transData, clip_on=False)
    axins.text(*ylabel_pos, ylabel, ha="center", va="center",
               fontsize=7, rotation=90, transform=axins.transData, clip_on=False)
    axins.text(-tick_spread, -tick_offset, x_low,  ha="left",  va="top",
               fontsize=6, transform=axins.transData, clip_on=False)
    axins.text(3 + tick_spread, -tick_offset, x_high, ha="right", va="top",
               fontsize=6, transform=axins.transData, clip_on=False)
    axins.text(-tick_offset, -tick_spread, y_low,  ha="right", va="bottom",
               fontsize=6, rotation=90, transform=axins.transData, clip_on=False)
    axins.text(-tick_offset, 3 + tick_spread, y_high, ha="right", va="top",
               fontsize=6, rotation=90, transform=axins.transData, clip_on=False)
    return axins


def _bv_render_map(gdf, state_gdf, color_col, ax,
                   colors_display=None, xlabel="", ylabel="",
                   y_low="Low", y_high="High", panel_label=None,
                   legend_bbox=(0.72, 0.01, 0.27, 0.34),
                   **legend_kwargs):
    """Plot bivariate choropleth with optional legend inset on *ax*.

    Extra keyword arguments (xlabel_pos, ylabel_pos, tick_spread, tick_offset)
    are forwarded directly to _bv_legend.
    """
    gdf.plot(ax=ax, color=gdf[color_col],
             edgecolor=COUNTY_EDGECOLOR, linewidth=0.15)
    state_gdf.plot(ax=ax, facecolor="none",
                   edgecolor=STATE_EDGECOLOR, linewidth=0.4)
    ax.set_xlim(MAP_XLIM)
    ax.set_ylim(MAP_YLIM)
    ax.set_aspect("equal")
    ax.axis("off")
    if panel_label is not None:
        ax.set_title(f"{panel_label})", loc="left", fontsize=8, fontweight="bold", pad=2)
    if colors_display is not None:
        _bv_legend(ax, colors_display, xlabel=xlabel, ylabel=ylabel,
                   y_low=y_low, y_high=y_high, bbox=legend_bbox,
                   **legend_kwargs)


def _bv_print_summary(gdf):
    print("  Tercile thresholds:")
    for col, name in [("eaua", "EAUA (weighted units/yr)"),
                      ("earp", "EARP (months/yr)"),
                      ("cc",   "CC (permits/month)")]:
        valid = gdf[col].dropna() if col in gdf.columns else pd.Series(dtype=float)
        valid = valid[valid > 0]
        if len(valid) > 0:
            print(f"    {name:35s}  p33={valid.quantile(1/3):.4f}"
                  f"  p67={valid.quantile(2/3):.4f}  n={len(valid)}")


def _bv_single_panel(gdf, state_gdf, color_col, grid, xlabel, ylabel,
                     y_low, y_high, stem,
                     legend_bbox=(0.667, 0.01, 0.33, 0.36),
                     **legend_kwargs):
    """Save one W_SINGLE bivariate map with its own legend inset.

    Extra keyword arguments (xlabel_pos, ylabel_pos, tick_spread, tick_offset)
    are forwarded to _bv_legend via _bv_render_map.
    """
    fig, ax = plt.subplots(1, 1, figsize=(W_SINGLE, 3))
    _bv_render_map(gdf, state_gdf, color_col, ax,
                   colors_display=grid,
                   xlabel=xlabel, ylabel=ylabel,
                   y_low=y_low, y_high=y_high,
                   legend_bbox=legend_bbox,
                   **legend_kwargs)
    plt.tight_layout()
    _save_fig(fig, stem)
    plt.close()


In [ ]:
gdf_bv = _bv_prepare_gdf(coastal_counties, eaua_df, earp_df, cc_df)
_bv_print_summary(gdf_bv)

_bv_single_panel(gdf_bv, state_gdf, "color_b", GRID_B,
                 xlabel="Risk", ylabel="CC",
                 y_low="High", y_high="Low",
                 xlabel_pos=(1.5, -1.5), ylabel_pos=(-1.8, 1.5),
                 tick_spread=0.4, tick_offset=0.8,
                 stem="bivariate_map_B_risk_vs_capacity")


## NRI construct validation

Construct validation of the permits-based construction capacity and the recovery
potential against the FEMA National Risk Index (NRI): convergent/discriminant
correlations, partial correlations controlling for exposure, the priority-county
screen, divergent counties, the combined SI table, and the SI figures.

This is NOT criterion validation (no observed recovery ground truth exists): we
test whether our metrics (i) move in the theoretically expected direction
relative to established resilience/vulnerability indices [convergent validity],
and (ii) are not redundant with them [discriminant validity].

Requires `data/NRI_Table_Counties.csv` (FEMA NRI county table, external download, see README).

In [ ]:
# ---- NRI analysis inputs / outputs ----
PERMITS      = DATA_DIR / "selected_states_counties_with_permits.csv"
F_EARP       = OUTPUT_DIR / "earp_per_county.csv"
F_DAMAGE     = OUTPUT_DIR / "county_event_frequency_damage_metrics.csv"
RECOVERY_CSV = DATA_DIR / "recovery" / "recovery_potential.csv"
IMPACT_DIR   = DATA_DIR / "impact" / "per_event"
NRI_COUNTY   = DATA_DIR / "NRI_Table_Counties.csv"
COUNTIES_SHP = DATA_DIR / "US_counties.shp"
OUT_DIR      = OUTPUT_DIR    # CSV results
FIG_DIR      = FIGURES_DIR   # SI figures
TAB_DIR      = TABLES_DIR    # SI LaTeX tables

CAP_COL = "Average_Building_Permits(12 months)"
FREQ = 1 / 1500  # events/yr, Poisson rate used in the study


def fips5(s):
    return (s.astype(str).str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^0-9]", "", regex=True).str.zfill(5))


def load_per_event_median():
    """Paper-style per-event recovery: median over ALL footprint events (incl.
    no-damage zeros) for each county; counties with zero capacity (NaN) dropped."""
    df = pd.read_csv(RECOVERY_CSV, dtype={"fips": str})
    df["fips"] = fips5(df["fips"])
    df["rec"] = pd.to_numeric(df["recovery_potential_months"], errors="coerce")
    finite = df[np.isfinite(df["rec"])]              # drop NaN (zero-capacity counties)
    agg = finite.groupby("fips")["rec"].agg(
        median_recovery_per_event="median", n_events="count").reset_index()
    print(f"  counties with finite recovery records: {len(agg)}")
    return agg


def load_nri_county():
    """Official FEMA NRI county table (no tract aggregation).

    HRCN_EALT is the Hurricane peril expected annual loss (essentially wind);
    CFLD_EALT is the separate Coastal Flooding peril EAL (storm surge). We load
    both so the hazard comparison can be checked against a wind-plus-surge match.
    """
    df = pd.read_csv(NRI_COUNTY, dtype={"STCOFIPS": str})
    df["fips"] = df["STCOFIPS"].str.zfill(5)
    nri_cols = ["RESL_SCORE", "SOVI_SCORE", "HRCN_EALT", "CFLD_EALT", "POPULATION"]
    # HRCN_EALB = hurricane building EAL (subset of HRCN_EALT; may not exist in all NRI versions)
    if "HRCN_EALB" in df.columns:
        nri_cols.append("HRCN_EALB")
    for c in nri_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    rename = {"RESL_SCORE": "nri_resilience", "SOVI_SCORE": "nri_social_vuln",
              "HRCN_EALT": "nri_hurricane_eal", "CFLD_EALT": "nri_coastal_flood_eal",
              "POPULATION": "nri_population"}
    if "HRCN_EALB" in df.columns:
        rename["HRCN_EALB"] = "nri_hurricane_eal_building"
    df = df.rename(columns=rename)
    keep = ["fips", "nri_population", "nri_resilience", "nri_social_vuln",
            "nri_hurricane_eal", "nri_coastal_flood_eal"]
    if "nri_hurricane_eal_building" in df.columns:
        keep.append("nri_hurricane_eal_building")
    return df[keep]


def load_inputs():
    earp = pd.read_csv(F_EARP); earp["fips"] = fips5(earp["fips"])
    dmg = pd.read_csv(F_DAMAGE); dmg["fips"] = fips5(dmg["fips"])
    dmg["eaua"] = dmg["total_weighted_damage_units"] * FREQ
    perm = pd.read_csv(PERMITS); perm["fips"] = fips5(perm["FIPS"])
    perm["construction_capacity"] = perm[CAP_COL]
    return earp, dmg, perm


def load_earc() -> pd.DataFrame:
    """Compute Expected Annual Repair Cost (EARC) per county from per_event CSVs.

    EARC_c = Σ_e repair_cost_sum_scaled_{e,c} * FREQ
    """
    files = sorted(IMPACT_DIR.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No per_event CSVs in {IMPACT_DIR}")
    print(f"Loading EARC from {len(files)} per_event files ...")
    df = pd.concat([pd.read_csv(f, usecols=["fips", "repair_cost_sum_scaled"]) for f in files],
                   ignore_index=True)
    earc = df.groupby("fips", as_index=False)["repair_cost_sum_scaled"].sum()
    earc["earc"] = earc["repair_cost_sum_scaled"] * FREQ
    earc["fips"] = fips5(earc["fips"])
    return earc[["fips", "earc"]]


def tercile(s):
    v = s.where((s > 0) & np.isfinite(s))
    return pd.qcut(v, 3, labels=[1, 2, 3])


def partial_spearman(df, x, y, z):
    """Spearman partial correlation of x,y controlling for z (rank-based)."""
    sub = df[[x, y, z]].replace([np.inf, -np.inf], np.nan).dropna()
    rx, ry, rz = sub[x].rank().values, sub[y].rank().values, sub[z].rank().values

    def resid(a, b):
        B = np.c_[np.ones(len(b)), b]
        coef, *_ = np.linalg.lstsq(B, a, rcond=None)
        return a - B @ coef
    ex, ey = resid(rx, rz), resid(ry, rz)
    r, p = pearsonr(ex, ey)
    return r, p, len(sub)


def _tex_signed(x, dagger=False):
    """Format a correlation with a LaTeX-safe minus and an optional dagger mark."""
    if pd.isna(x):
        return "--"
    s = ("$-$" if x < 0 else "+") + f"{abs(x):.2f}"
    return s + (r"$^{\dagger}$" if dagger else "")


def write_latex_tables(out_dir, risk, corr, partial, si, n_sample):
    """Write the two SI tables: combined NRI evaluation, and divergent counties."""
    cset = corr.set_index(["our_metric", "nri_metric"])
    rrow = {r["comparison"]: r for _, r in risk.iterrows()}
    pset = partial.set_index("our_metric")

    nri_cols = ["Community Resilience", "Social Vulnerability", "Hurricane EAL (exposure)"]
    rec_rows = [("Construction capacity", "capacity"),
                ("EARP (annual recovery potential)", "EARP (annual recovery potential)"),
                ("Median per-event recovery potential", "median per-event recovery potential")]

    L = [r"\begin{table}[htb!]", r"\centering",
         (r"\caption{Evaluation of the recovery-potential model against the FEMA National "
          r"Risk Index (NRI), over the analysis sample of " + str(n_sample) + r" US "
          r"Atlantic-coast counties with positive modelled risk, recovery potential and "
          r"construction capacity. Panel~A compares the modelled hazard layer (expected "
          r"annual units affected, EAUA; expected annual repair cost, EARC) with the "
          r"NRI's independent, historically calibrated monetary risk. Panel~B reports "
          r"rank correlations between the model's quantities and the NRI socio-economic "
          r"layers (convergent and discriminant validity). Panel~C re-estimates the "
          r"association with Social Vulnerability after controlling for exposure. Entries "
          r"are Spearman rank coefficients; Panel~A also gives Pearson coefficients on "
          r"$\log_{10}$-transformed values. All coefficients are significant at $p<0.01$ "
          r"unless marked.}"),
         r"\label{tab:nri_evaluation}",
         r"\begin{tabular}{lrrr}", r"\hline",
         (r"\multicolumn{4}{l}{\textit{Panel A. Hazard layer: modelled physical risk vs "
          r"independent monetary risk}}\\"),
         r" & Spearman & Pearson (log) & $n$ \\", r"\hline"]
    for key in ["EAUA vs NRI hurricane (wind) EAL", "EAUA vs NRI coastal-flood EAL",
                "EAUA vs NRI wind + coastal-flood EAL"]:
        r = rrow.get(key)
        if r is None:
            continue
        L.append(f"{key} & {_tex_signed(r['spearman_r'])} & "
                 f"{_tex_signed(r['pearson_log_r'])} & {int(r['n'])} \\\\")
    # EARC rows (added if available)
    for key in list(rrow.keys()):
        if not key.startswith("EARC"):
            continue
        r = rrow[key]
        L.append(f"{key} & {_tex_signed(r['spearman_r'])} & "
                 f"{_tex_signed(r['pearson_log_r'])} & {int(r['n'])} \\\\")
    L += [r"\hline",
          (r"\multicolumn{4}{l}{\textit{Panel B. Recovery layer vs NRI socio-economic "
           r"indices (Spearman $r$)}}\\"),
          r" & Community & Social & Hurricane \\",
          r" & Resilience & Vulnerability & EAL (exposure) \\", r"\hline"]
    for disp, key in rec_rows:
        vals = [_tex_signed(cset.loc[(key, n), "spearman_r"],
                            dagger=float(cset.loc[(key, n), "p"]) > 0.05) for n in nri_cols]
        L.append(f"{disp} & " + " & ".join(vals) + r" \\")
    L += [r"\hline",
          (r"\multicolumn{4}{l}{\textit{Panel C. Association with Social Vulnerability, "
           r"controlling for exposure (Spearman $r$)}}\\"),
          r" & raw & partial & partial \\",
          r" & & (control EAUA) & (control wind EAL) \\", r"\hline"]
    for disp, key in [("Construction capacity", "capacity"),
                      ("Median per-event recovery potential", "median per-event recovery")]:
        p = pset.loc[key]
        L.append(f"{disp} & {_tex_signed(p['spearman_raw'])} & "
                 f"{_tex_signed(p['partial_given_EAUA'])} & "
                 f"{_tex_signed(p['partial_given_hurricaneEAL'])} \\\\")
    L += [r"\hline", r"\end{tabular}", r"\\[2pt]",
          (r"{\footnotesize $^{\dagger}$ Not significant ($p>0.05$): per-event recovery "
           r"potential is statistically independent of hazard exposure, unlike the annual "
           r"metric.}"),
          r"\end{table}"]
    (out_dir / "table_nri_evaluation.tex").write_text("\n".join(L) + "\n")

    D = [r"\begin{table}[htb!]", r"\centering",
         (r"\caption{Priority counties that socio-economic screening would miss. A "
          r"priority county is one ranked in both the highest tercile of expected "
          r"tropical cyclone (TC) damage and the lowest tercile of construction capacity "
          r"across the analysis sample ($n = 685$ counties); these are places where high "
          r"storm risk coincides with a thin local rebuilding sector. A county is "
          r"considered flagged by a socio-economic index when it falls in that index's "
          r"most concerning tercile, that is, the highest tercile of the FEMA National "
          r"Risk Index (NRI) Social Vulnerability score (SVI) or the lowest tercile of "
          r"the NRI Community Resilience score (Resil.). The 13 counties listed are "
          r"priority counties in neither of those terciles, so a screen built on either "
          r"NRI index would overlook them; they are the counties where the "
          r"building-permit capacity screen adds information beyond socio-economic "
          r"indicators. Counties are sorted by number of housing units. Pop.\ \% of "
          r"state is the percentage of the state population living in the county. Permits "
          r"per month is the construction-capacity proxy (average monthly building "
          r"permits), and permits per 1000 units normalises it by housing stock (sample "
          r"median 0.046). NRI SVI and Community Resilience are county percentile scores "
          r"from 0 to 100; a higher SVI indicates greater social vulnerability and a "
          r"higher Resilience score indicates greater resilience.}"),
         r"\label{tab:divergent_counties}",
         r"\begin{tabular}{llrrrrrr}", r"\hline",
         r"County & State & Housing & Pop.\ \% & Permits & Permits per & NRI & NRI \\",
         r" & & units & of state & per month & 1000 units & SVI & Resil. \\", r"\hline"]
    for _, r in si.iterrows():
        D.append(f"{r['county']} & {r['state']} & {int(round(r['housing_units'])):,} & "
                 f"{r['pop_pct_of_state']:.2f} & {r['permits_per_month']:.2f} & "
                 f"{r['permits_per_1000_units']:.3f} & {int(round(r['nri_social_vuln']))} & "
                 f"{int(round(r['nri_resilience']))} \\\\")
    D += [r"\hline", r"\end{tabular}", r"\end{table}"]
    (out_dir / "table_divergent_counties.tex").write_text("\n".join(D) + "\n")


In [ ]:
print("Loading raw pyrecodes per-event recovery ...")
per_event = load_per_event_median()
earp, dmg, perm = load_inputs()
nri = load_nri_county()
earc_df = load_earc()

m = (earp.merge(dmg[["fips", "eaua", "total_weighted_damage_units", "total_units"]], on="fips", how="outer")
          .merge(perm[["fips", "NAME", "STATE_NAME", "construction_capacity"]], on="fips", how="left")
          .merge(per_event, on="fips", how="left")
          .merge(nri, on="fips", how="left")
          .merge(earc_df, on="fips", how="left"))

# analysis sample: positive EARP / EAUA / capacity (matches the paper)
m["in_sample"] = (m["earp_months_per_year"] > 0) & (m["eaua"] > 0) & (m["construction_capacity"] > 0)
s = m[m["in_sample"]].copy()

# priority screen (Map B): EAUA tercile 3 x CC tercile 1
s["eaua_t"] = tercile(s["eaua"])
s["cc_t"] = tercile(s["construction_capacity"])
s["earp_t"] = tercile(s["earp_months_per_year"])
s["priority"] = (s["eaua_t"] == 3) & (s["cc_t"] == 1)
m = m.merge(s[["fips", "eaua_t", "cc_t", "earp_t", "priority"]], on="fips", how="left")
OUT_DIR.mkdir(exist_ok=True)
m.to_csv(OUT_DIR / "nri_county_merged.csv", index=False)
print(f"\nAnalysis sample: {len(s)} counties | priority (EAUA t3 x CC t1): {int(s['priority'].sum())}")


In [ ]:
# ---- HAZARD LAYER: modelled physical risk vs independent monetary risk ----
# Our EAUA (expected annual units affected, units/yr) is a physical risk metric;
# the NRI hurricane EAL is a historically calibrated monetary risk ($) built from
# entirely separate inputs. Agreement corroborates the hazard layer without sharing
# data. NRI books storm surge under a separate Coastal Flooding peril, so we also
# report EAUA against coastal-flood EAL and against wind+coastal-flood as a
# peril-matching robustness check.
def corr_pair(x, y):
    d = s[[x, y]].replace([np.inf, -np.inf], np.nan)
    d = d[(d[x] > 0) & (d[y] > 0)].dropna()
    sp, sp_p = spearmanr(d[x], d[y])
    pe = pearsonr(np.log10(d[x]), np.log10(d[y]))[0]
    return sp, pe, sp_p, len(d)

s["nri_wind_plus_cflood_eal"] = s[["nri_hurricane_eal", "nri_coastal_flood_eal"]].sum(
    axis=1, min_count=1)
risk_rows = []
for yc, yl in [("nri_hurricane_eal", "NRI hurricane (wind) EAL"),
               ("nri_coastal_flood_eal", "NRI coastal-flood EAL"),
               ("nri_wind_plus_cflood_eal", "NRI wind + coastal-flood EAL")]:
    sp, pe, sp_p, n = corr_pair("eaua", yc)
    risk_rows.append({"metric": "EAUA", "comparison": f"EAUA vs {yl}",
                      "spearman_r": round(sp, 3),
                      "pearson_log_r": round(pe, 3), "p": f"{sp_p:.1e}", "n": n})
# EARC vs NRI EAL (building EAL preferred; fall back to hurricane total EAL)
earc_nri_cols = []
if "nri_hurricane_eal_building" in s.columns:
    earc_nri_cols.append(("nri_hurricane_eal_building", "NRI hurricane building EAL"))
earc_nri_cols += [("nri_hurricane_eal", "NRI hurricane (wind) EAL"),
                  ("nri_wind_plus_cflood_eal", "NRI wind + coastal-flood EAL")]
for yc, yl in earc_nri_cols:
    if yc not in s.columns:
        continue
    sp, pe, sp_p, n = corr_pair("earc", yc)
    risk_rows.append({"metric": "EARC", "comparison": f"EARC vs {yl}",
                      "spearman_r": round(sp, 3),
                      "pearson_log_r": round(pe, 3), "p": f"{sp_p:.1e}", "n": n})
risk = pd.DataFrame(risk_rows)
risk.to_csv(OUT_DIR / "nri_risk_comparison.csv", index=False)
print("\nHazard layer (EAUA/EARC vs NRI EAL):")
print(risk.to_string(index=False))

# internal consistency (both statistics, for the record)
print("\nInternal consistency (analysis sample):")
for a, b in [("earp_months_per_year", "eaua"), ("earp_months_per_year", "construction_capacity"),
             ("median_recovery_per_event", "construction_capacity"),
             ("median_recovery_per_event", "eaua")]:
    d = s[[a, b]].replace([np.inf, -np.inf], np.nan).dropna()
    sp = spearmanr(d[a], d[b])[0]
    dlog = d[(d[a] > 0) & (d[b] > 0)]
    pe = pearsonr(np.log10(dlog[a]), np.log10(dlog[b]))[0] if len(dlog) >= 3 else float("nan")
    print(f"  {a:26s} vs {b:22s}: Spearman={sp:+.3f}  Pearson(log)={pe:+.3f}  n={len(d)}")


In [ ]:
# ---- discriminant correlations: our metrics vs NRI ----
our = [("construction_capacity", "capacity"),
       ("earp_months_per_year", "EARP (annual recovery potential)"),
       ("median_recovery_per_event", "median per-event recovery potential")]
nri_cols = [("nri_resilience", "Community Resilience"),
            ("nri_social_vuln", "Social Vulnerability"),
            ("nri_hurricane_eal", "Hurricane EAL (exposure)")]
rows = []
for xc, xl in our:
    for yc, yl in nri_cols:
        d = s[[xc, yc]].replace([np.inf, -np.inf], np.nan).dropna()
        r, p = spearmanr(d[xc], d[yc])
        rows.append({"our_metric": xl, "nri_metric": yl,
                     "spearman_r": round(r, 3), "p": f"{p:.1e}", "n": len(d)})
corr = pd.DataFrame(rows)
corr.to_csv(OUT_DIR / "nri_correlations.csv", index=False)
print("\nDiscriminant correlations (Spearman, our metrics vs NRI):")
print(corr.to_string(index=False))


In [ ]:
# ---- partial correlations: control for exposure ----
# two exposure controls: our own model exposure (EAUA) and NRI hurricane EAL ($)
prows = []
for xc, xl in [("median_recovery_per_event", "median per-event recovery"),
               ("construction_capacity", "capacity")]:
    d0 = s[[xc, "nri_social_vuln"]].replace([np.inf, -np.inf], np.nan).dropna()
    r0 = spearmanr(d0[xc], d0["nri_social_vuln"])[0]
    r_eaua = partial_spearman(s, xc, "nri_social_vuln", "eaua")[0]
    r_heal = partial_spearman(s, xc, "nri_social_vuln", "nri_hurricane_eal")[0]
    prows.append({"our_metric": xl, "vs": "Social Vulnerability",
                  "spearman_raw": round(r0, 3),
                  "partial_given_EAUA": round(r_eaua, 3),
                  "partial_given_hurricaneEAL": round(r_heal, 3), "n": len(d0)})
partial = pd.DataFrame(prows)
partial.to_csv(OUT_DIR / "nri_partial_correlations.csv", index=False)
print("\nPartial correlation (controlling for hurricane EAL exposure):")
print(partial.to_string(index=False))


In [ ]:
# ---- priority counties vs NRI terciles ----
s["resl_t"] = tercile(s["nri_resilience"])
s["sovi_t"] = tercile(s["nri_social_vuln"])
pr = s[s["priority"]]
n_pri = len(pr)
ct_resl = pd.crosstab(pr["priority"], pr["resl_t"])
ct_sovi = pd.crosstab(pr["priority"], pr["sovi_t"])
pd.concat([ct_resl.rename(index={True: "resilience"}),
           ct_sovi.rename(index={True: "social_vuln"})]).to_csv(OUT_DIR / "nri_priority_crosstab.csv")
print(f"\nPriority counties (n={n_pri}) by NRI Community Resilience tercile (1=low,3=high):")
print(pr["resl_t"].value_counts().sort_index().to_string())
print(f"Priority counties by NRI Social Vulnerability tercile (1=low,3=high):")
print(pr["sovi_t"].value_counts().sort_index().to_string())
hi_sovi = (pr["sovi_t"] == 3).mean() * 100
lo_resl = (pr["resl_t"] == 1).mean() * 100
print(f"\nHeadline: {hi_sovi:.0f}% of priority counties are top-tercile SVI; "
      f"{100-hi_sovi:.0f}% are NOT.")
print(f"          {lo_resl:.0f}% are bottom-tercile resilience; {100-lo_resl:.0f}% are NOT.")


In [ ]:
# ---- divergent counties: priority but flagged by NEITHER index ----
s["divergent"] = s["priority"] & ~((s["sovi_t"] == 3) | (s["resl_t"] == 1))
state_pop = nri.groupby(nri["fips"].str[:2])["nri_population"].sum()
div = s[s["divergent"]].copy()
div["pop_pct_state"] = div.apply(
    lambda r: 100 * r["nri_population"] / state_pop[r["fips"][:2]], axis=1)
div["permits_per_1000units"] = div["construction_capacity"] / div["total_units"] * 1000
si = div[["fips", "NAME", "STATE_NAME", "total_units", "pop_pct_state",
          "construction_capacity", "permits_per_1000units",
          "nri_social_vuln", "nri_resilience"]].sort_values("total_units")
si.columns = ["fips", "county", "state", "housing_units", "pop_pct_of_state",
              "permits_per_month", "permits_per_1000_units", "nri_social_vuln",
              "nri_resilience"]
si.to_csv(OUT_DIR / "divergent_counties_SI.csv", index=False)
print(f"\nDivergent priority counties (flagged by neither index): {len(div)}")
print(f"  median housing units: {div['total_units'].median():.0f} "
      f"(study median {s['total_units'].median():.0f})")
print(f"  median permits/1000 units: {div['permits_per_1000units'].median():.3f} "
      f"(study median {(s['construction_capacity']/s['total_units']*1000).median():.3f})")


In [ ]:
# ---- SI tables (combined NRI evaluation + divergent counties) ----
write_latex_tables(TAB_DIR, risk, corr, partial, si, len(s))
print("\nWrote SI tables: table_nri_evaluation.tex, table_divergent_counties.tex")


In [ ]:
# ---- SI figure: EAUA vs NRI hurricane (wind) EAL ----
dfr = s[["NAME", "STATE_NAME", "eaua", "nri_hurricane_eal"]].copy()
dfr = dfr[(dfr["eaua"] > 0) & (dfr["nri_hurricane_eal"] > 0)].dropna()
dfr["gap"] = dfr["eaua"].rank(pct=True) - dfr["nri_hurricane_eal"].rank(pct=True)
figr, axr = plt.subplots(figsize=(6.5, 6))
axr.scatter(dfr["nri_hurricane_eal"], dfr["eaua"], s=14, c="#9ecae1",
            edgecolor="none", alpha=0.7)
label = pd.concat([dfr.nlargest(4, "gap"), dfr.nsmallest(4, "gap")])
axr.scatter(label["nri_hurricane_eal"], label["eaua"], s=24, c="#d7191c", edgecolor="none")
for _, r in label.iterrows():
    axr.annotate(r["NAME"], (r["nri_hurricane_eal"], r["eaua"]), fontsize=7,
                 xytext=(3, 3), textcoords="offset points", color="#d7191c")
axr.set_xscale("log"); axr.set_yscale("log")
axr.set_xlabel("NRI hurricane (wind) expected annual loss (USD)")
axr.set_ylabel("EAUA (expected annual units affected, units/yr)")
rho = spearmanr(dfr["eaua"], dfr["nri_hurricane_eal"])[0]
axr.set_title(f"Modelled physical risk vs NRI monetary risk "
              f"(Spearman $r={rho:+.2f}$, $n={len(dfr)}$)", fontsize=9)
figr.tight_layout()
figr.savefig(FIG_DIR / "fig_eaua_vs_nri_eal.png", dpi=200, bbox_inches="tight")


In [ ]:
# ---- SI figure: 2x2 agreement maps, recovery potential vs NRI socio-economic indices ----
# Both quantities are oriented as "expected recovery difficulty" via within-sample
# percentile ranks (n = analysis sample). The mapped value is the recovery-difficulty
# rank minus the NRI-difficulty rank: red (positive) = our recovery metric flags the
# county as harder to recover than the NRI score does (physical-capacity blind spots);
# blue (negative) = the NRI score flags greater difficulty (more vulnerable / less
# resilient) than our recovery does; white = the two agree.
COASTAL = ['01', '09', '10', '12', '13', '22', '23', '24', '25', '28',
           '33', '34', '36', '37', '42', '44', '45', '48', '51']
geo = gpd.read_file(COUNTIES_SHP)
geo["fips"] = (geo["STATEFP"] + geo["COUNTYFP"]).str.zfill(5)
geo = geo[geo["STATEFP"].isin(COASTAL)].copy()

def rank_pct(col, higher_is_harder=True):
    r = s[col].replace([np.inf, -np.inf], np.nan).rank(pct=True)
    return r if higher_is_harder else (1.0 - r)

# recovery metrics: higher value = slower recovery = harder
s["rk_earp"] = rank_pct("earp_months_per_year")
s["rk_medrec"] = rank_pct("median_recovery_per_event")
# NRI: higher SVI = harder; lower Resilience = harder
s["rk_svi"] = rank_pct("nri_social_vuln", higher_is_harder=True)
s["rk_resl"] = rank_pct("nri_resilience", higher_is_harder=False)

rec_rows = [("rk_earp", "earp_months_per_year", "EARP (annual recovery potential)"),
            ("rk_medrec", "median_recovery_per_event", "Median per-event recovery potential")]
nri_cols2 = [("rk_resl", "Community Resilience"),
             ("rk_svi", "Social Vulnerability")]

cmap = plt.cm.RdBu_r
norm = plt.Normalize(vmin=-1, vmax=1)
fig2, axes = plt.subplots(2, 2, figsize=(11, 8.2))
for i, (rk_rec, _, rec_lbl) in enumerate(rec_rows):
    for j, (rk_nri, nri_lbl) in enumerate(nri_cols2):
        ax = axes[i][j]
        s["agree"] = s[rk_rec] - s[rk_nri]
        g = geo.merge(s[["fips", "agree"]], on="fips", how="left")
        g.plot(ax=ax, color="#e9e9e9", edgecolor="#cccccc", linewidth=0.1)   # base / off-sample
        gv = g[g["agree"].notna()]
        gv.plot(ax=ax, column="agree", cmap=cmap, norm=norm,
                edgecolor="#888888", linewidth=0.1)
        ax.set_xlim(-107, -65); ax.set_ylim(24, 48); ax.set_aspect("equal"); ax.axis("off")
        rho = spearmanr(s[rk_rec], s[rk_nri], nan_policy="omit")[0]
        ax.set_title(f"{rec_lbl}\nvs NRI {nri_lbl}  (agreement $r={rho:+.2f}$)", fontsize=9)
fig2.subplots_adjust(left=0.02, right=0.88, top=0.90, bottom=0.04, wspace=0.04, hspace=0.14)
cax = fig2.add_axes([0.90, 0.22, 0.016, 0.56])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cb = fig2.colorbar(sm, cax=cax)
cb.set_label("recovery-difficulty rank  $-$  NRI-difficulty rank\n"
             "red: model flags worse        blue: NRI flags worse", fontsize=8)
fig2.suptitle("Agreement between modelled recovery potential and FEMA NRI "
              "socio-economic indices\n(within-sample percentile ranks, both oriented "
              "as expected recovery difficulty)", fontsize=11)
fig2.savefig(FIG_DIR / "fig_recovery_nri_agreement_2x2.png", dpi=200, bbox_inches="tight")


In [ ]:
# ---- SI figure: single panel, median per-event recovery vs NRI Community Resilience ----
# Same rank-difference construction as the 2x2; the 13 divergent priority counties
# (Table tab:divergent_counties) are outlined in black.
s["agree_mr_resl"] = s["rk_medrec"] - s["rk_resl"]
g1 = geo.merge(s[["fips", "agree_mr_resl", "divergent"]], on="fips", how="left")
fig1, ax1 = plt.subplots(figsize=(7.5, 6))
g1.plot(ax=ax1, color="#e9e9e9", edgecolor="#cccccc", linewidth=0.1)   # base / off-sample
g1[g1["agree_mr_resl"].notna()].plot(ax=ax1, column="agree_mr_resl", cmap=cmap, norm=norm,
                                     edgecolor="#888888", linewidth=0.1)
dvg = g1[g1["divergent"] == True]
if len(dvg):
    dvg.plot(ax=ax1, facecolor="none", edgecolor="black", linewidth=1.1)
ax1.set_xlim(-107, -65); ax1.set_ylim(24, 48); ax1.set_aspect("equal"); ax1.axis("off")
cax1 = fig1.add_axes([0.84, 0.24, 0.018, 0.52])
sm1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm1.set_array([])
cb1 = fig1.colorbar(sm1, cax=cax1)
cb1.set_label("recovery-difficulty rank  $-$  resilience-difficulty rank", fontsize=8)
cb1.ax.text(0.5, 1.03, "red: model flags worse", transform=cb1.ax.transAxes,
            fontsize=8, ha="center", va="bottom")
cb1.ax.text(0.5, -0.03, "blue: NRI flags worse", transform=cb1.ax.transAxes,
            fontsize=8, ha="center", va="top")
fig1.savefig(FIG_DIR / "fig_recovery_resilience_agreement.png", dpi=200, bbox_inches="tight")


In [ ]:
# ---- figure ----
fig, ax = plt.subplots(figsize=(7, 5.5))
base = s[~s["priority"]]
ax.scatter(base["nri_resilience"], base["construction_capacity"], s=14,
           c="#bdbdbd", alpha=0.6, edgecolor="none", label="other counties")
ax.scatter(pr["nri_resilience"], pr["construction_capacity"], s=30,
           c="#d7191c", alpha=0.85, edgecolor="none", label="priority (high risk / low capacity)")
ax.set_yscale("log")
ax.set_xlabel("NRI Community Resilience score")
ax.set_ylabel("Construction capacity (permits/month, log)")
ax.set_title("Priority counties vs NRI Community Resilience")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_capacity_vs_resilience.png", dpi=150)
print("\nWrote merged CSV, correlations, partial correlations, crosstab, figure.")
